## Function to input images

In [1]:

"""
el_blocks.py
============
 
Building blocks for module-level EL analysis (Rajput et al. 2018), to be called
from a larger run_module_analysis(). Three blocks so far:
 
    1. mask_cell_pair(low, high)  -> (phi_low, phi_high, mask)   [cv2 + Otsu]
    2. compute_f(low_cells, ...)  -> f, S        (one module constant)
    3. compute_Vi(S or cells, f)  -> Vi per cell
 
Formulae (your screenshots):
 
    f   = exp{ [ Σ_i (U_Th/2)·ln( I / S_i ) − V_T ] / ( (N/2)·U_Th ) }     (Eq.16)
    V_i = (U_Th/2)·ln( I / ( f · S_i ) )                                    (Eq.13)
    with  S_i = ∫ (1/Φ(r)) d²r  ≈  a_pixel · Σ_pixels (1/Φ_low)
 
The "30" in the paper's f is N/2 with N = 60, kept general here as N/2.
 
================================ IMPORTANT cv2 TRAP ============================
cv2.imread(path) WITHOUT a flag downconverts a 16-bit TIFF to 8-bit BGR. That
silently destroys the linear intensity scale the whole method depends on. Always
read with IMREAD_UNCHANGED (done in load_gray below).
 
Also: cv2's Otsu (THRESH_OTSU) only accepts 8-bit images. So the threshold is
found on an 8-bit *normalised copy*, but the mask is applied to the original
float data — the Φ values used downstream are never quantised to 8-bit.
==============================================================================
"""
 
from __future__ import annotations
import os
import glob
import numpy as np
import cv2
 
 
# ----------------------------------------------------------------- I/O
def load_gray(path_or_array) -> np.ndarray:
    """Read an EL image preserving bit depth; return a float64 2D array."""
    if not isinstance(path_or_array, str):
        return np.asarray(path_or_array, dtype=np.float64)
    img = cv2.imread(path_or_array, cv2.IMREAD_UNCHANGED)   # keep 16-bit!
    if img is None:
        raise FileNotFoundError(f"cv2 could not read: {path_or_array}")
    if img.ndim == 3:                                       # color -> gray, keep depth
        img = cv2.cvtColor(img[..., :3], cv2.COLOR_BGR2GRAY)
    return img.astype(np.float64)
 
 
# ----------------------------------------------------------------- 1) MASKING
def mask_cell_pair(low, high, *, min_valid_intensity: float = 1e-6,
                   mask_from: str = "low"):
    """Otsu-mask a low/high cell pair with ONE consistent mask.
 
    Parameters
    ----------
    low, high : str path or 2D array (one cell crop each)
    min_valid_intensity : floor; pixels at/below this are excluded so 1/Φ is finite
    mask_from : "low" (default), "high", or "both" (intersection of each image's
                Otsu mask). "low" is usually best — low bias is more uniform.
 
    Returns
    -------
    phi_low, phi_high : float64 2D arrays (same shape)
    mask : bool 2D array, applied identically to both
    """
    phi_low = load_gray(low)
    phi_high = load_gray(high)
    if phi_high.shape != phi_low.shape:
        phi_high = cv2.resize(phi_high, (phi_low.shape[1], phi_low.shape[0]),
                              interpolation=cv2.INTER_NEAREST)
 
    def otsu_mask(img):
        # cv2 Otsu needs 8-bit: normalise a COPY just to find the threshold
        norm8 = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        thr8, _ = cv2.threshold(norm8, 0, 255,
                                cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        return norm8 > thr8
 
    if mask_from == "low":
        mask = otsu_mask(phi_low)
    elif mask_from == "high":
        mask = otsu_mask(phi_high)
    else:                                   # "both"
        mask = otsu_mask(phi_low) & otsu_mask(phi_high)
 
    mask &= (phi_low > min_valid_intensity) & (phi_high > min_valid_intensity)
    return phi_low, phi_high, mask
 
 
# ----------------------------------------------------------------- S_i
def cell_Si(phi_low: np.ndarray, mask: np.ndarray,
            cell_area_cm2: float | None = None) -> float:
    """S_i = ∫(1/Φ)d²r over the cell. a_pixel = cell_area/n_active (or 1 if None).
 
    Vi and Rs are invariant to this area choice (it cancels through f); it only
    sets the absolute scale of J0 [fA/cm²]. Pass cell_area_cm2 (e.g. 15.6**2)
    when you want calibrated J0, leave None for relative units.
    """
    n = int(mask.sum())
    if n == 0:
        return np.nan
    a_pixel = (cell_area_cm2 / n) if cell_area_cm2 else 1.0
    return a_pixel * float(np.sum(1.0 / phi_low[mask]))

## calculating variables: f and Vi

In [2]:
# ----------------------------------------------------------------- 2) f
def compute_f(low_cells, I_low: float, VT: float, Uth: float,
              N: int | None = None, cell_area_cm2: float | None = None):
    """Module constant f (Eq. 16).
 
    low_cells : list of (phi_low, mask) for every cell in the module
    Returns (f, S) where S is the array of per-cell S_i (reuse it for Vi).
    """
    S = np.array([cell_Si(p, m, cell_area_cm2) for p, m in low_cells], dtype=float)
    if not np.all(np.isfinite(S)) or np.any(S <= 0):
        raise ValueError("invalid S_i (empty mask or non-positive intensity)")
    if N is None:
        N = len(S)
    numerator = np.sum((Uth / 2.0) * np.log(I_low / S)) - VT
    ln_f = numerator / ((N / 2.0) * Uth)        # paper's 30·U_Th == (N/2)·U_Th
    return float(np.exp(ln_f)), S

# ----------------------------------------------------------------- 3) Vi
def compute_Vi(low_cells_or_S, f: float, I_low: float, Uth: float,
               cell_area_cm2: float | None = None) -> np.ndarray:
    """Per-cell terminal voltage at low bias (Eq. 13):
 
        V_i = (U_Th/2)·ln( I_low / (f · S_i) )
 
    Accepts either the S array from compute_f, or the list of (phi_low, mask).
    """
    if isinstance(low_cells_or_S, np.ndarray):
        S = low_cells_or_S.astype(float)
    elif (isinstance(low_cells_or_S, (list, tuple)) and low_cells_or_S
          and isinstance(low_cells_or_S[0], (int, float, np.floating))):
        S = np.asarray(low_cells_or_S, dtype=float)
    else:
        S = np.array([cell_Si(p, m, cell_area_cm2) for p, m in low_cells_or_S],
                     dtype=float)
    return (Uth / 2.0) * np.log(I_low / (f * S))



## Maps from low bias images: C_map and J0_map

In [3]:
# ----------------------------------------------------------------- 3) c(r)
def compute_c_map(phi_low, Vi, Uth: float = 0.0259, mask=None):
    """c(r) = Φ_low(r) · exp(−Vᵢ/U_Th)   (Eq. 8).
 
    Pass the mask from mask_cell_pair so c(r) uses the same pixels as f/Vi.
    Returns (c_map, mask), c_map = NaN outside the mask.
    """
    phi = np.asarray(phi_low, dtype=np.float64)
    if mask is None:
        mask = np.isfinite(phi) & (phi > 0)
    c_map = np.full_like(phi, np.nan)
    c_map[mask] = phi[mask] * np.exp(-float(Vi) / float(Uth))
    return c_map, mask
 
# ----------------------------------------------------------------- 4) J0(r)
def compute_J0_map(c_map, f: float, mask=None):
    """J0(r) = f / c(r)   (Eq. 3/9).  Per-pixel units (relative J0)."""
    if f <= 0:
        raise ValueError("f must be > 0")
    J0 = np.full_like(c_map, np.nan, dtype=np.float64)
    valid = np.isfinite(c_map) & (c_map > 0)
    if mask is not None:
        valid &= mask
    J0[valid] = float(f) / c_map[valid]
    return J0
 
 
def map_scalar(arr, mask=None, stat: str = "mean") -> float:
    """Reduce a per-cell map to a scalar over its mask (paper: J0,i = mean)."""
    sel = (mask if mask is not None else np.isfinite(arr)) & np.isfinite(arr)
    v = arr[sel]
    if v.size == 0:
        return np.nan
    fn = {"mean": np.mean, "median": np.median, "min": np.min, "max": np.max}[stat]
    return float(fn(v))
    
# ------------------------------------------------- module mosaic (display/save)
def assemble_module_map(cell_maps, n_rows: int = 6, n_cols: int = 10, *,
                        order: str = "row", gap: int = 2, bg: float = np.nan):
    """Tile per-cell maps into one module image.
 
    order : "row"  index 0 = top-left, fill left→right, top→bottom
            "col"  fill top→bottom, then next column
            "snake" boustrophedon (row 0 L→R, row 1 R→L, …) — common in modules
    If the mosaic looks scrambled, switch order to match how the crops were
    indexed. Cells are top-left aligned in equal tiles (max cell size).
    """
    cell_maps = list(cell_maps)
    H = max(m.shape[0] for m in cell_maps)
    W = max(m.shape[1] for m in cell_maps)
    canvas = np.full((n_rows * H + (n_rows - 1) * gap,
                      n_cols * W + (n_cols - 1) * gap), bg, dtype=np.float64)
    for i, m in enumerate(cell_maps):
        if order == "row":
            r, c = divmod(i, n_cols)
        elif order == "col":
            c, r = divmod(i, n_rows)
        elif order == "snake":
            r, c = divmod(i, n_cols)
            if r % 2 == 1:
                c = n_cols - 1 - c
        else:
            raise ValueError("order must be 'row', 'col', or 'snake'")
        if r >= n_rows or c >= n_cols:
            continue
        y, x = r * (H + gap), c * (W + gap)
        canvas[y:y + m.shape[0], x:x + m.shape[1]] = m
    return canvas
 
 
def display_module_map(mosaic, title: str = "", cmap: str = "viridis",
                       clip=(2, 98), figsize=(12, 6)):
    """Show a mosaic with a percentile-clipped colour scale (NaN gaps blank)."""
    import matplotlib.pyplot as plt
    finite = mosaic[np.isfinite(mosaic)]
    vmin, vmax = (np.percentile(finite, clip) if finite.size else (0.0, 1.0))
    plt.figure(figsize=figsize)
    im = plt.imshow(mosaic, cmap=cmap, vmin=vmin, vmax=vmax)
    plt.colorbar(im, fraction=0.025)
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.show()
 
 
def save_module_map_tiff(mosaic, path: str):
    """Save a float mosaic as a 32-bit TIFF (NaN preserved). Needs .tif/.tiff."""
    ok = cv2.imwrite(path, mosaic.astype(np.float32))
    if not ok:
        raise IOError(f"cv2 failed to write {path} (use a .tif/.tiff extension)")
 
 
# --------------------------------------- full low-bias branch (Steps 1–4)
def compute_low_bias_maps(low_folder, high_folder, I_low, VT, *,
                          Uth: float = 0.0259, N: int = 60, ext: str = ".tiff",
                          cell_area_cm2=None, min_valid_intensity: float = 1e-6):
    """Run Steps 1–4 for one module and return f, Vi, and the per-cell maps.
 
    Returns dict: f, S, Vi, c_maps, J0_maps, J0_i (per-cell mean J0).
    Feed J0_maps to assemble_module_map(...) to display/save the module J0 map.
    """
    cells = load_module_low_cells(low_folder, ext, high_folder=high_folder,
                                  min_valid_intensity=min_valid_intensity)
    f, S = compute_f(cells, I_low, VT, Uth, N=N, cell_area_cm2=cell_area_cm2)
    Vi = compute_Vi(S, f, I_low, Uth)
    c_maps, J0_maps, J0_i = [], [], []
    for (phi, mask), vi in zip(cells, Vi):
        c_map, _ = compute_c_map(phi, vi, Uth, mask=mask)
        J0 = compute_J0_map(c_map, f, mask=mask)
        c_maps.append(c_map)
        J0_maps.append(J0)
        J0_i.append(map_scalar(J0, mask, "mean"))
    return {"f": f, "S": S, "Vi": Vi, "c_maps": c_maps,
            "J0_maps": J0_maps, "J0_i": np.asarray(J0_i)}



## High bias 

In [4]:
# =============================================================== HIGH BIAS
# Steps 5-8. These need the high-bias masked image plus the LOW-bias c(r) and
# J0(r). Note: Rs is independent of f and V_T (both cancel), so this branch is
# robust even if the low-bias terminal-voltage calibration is uncertain.
 
# ----------------------------------------------------------------- 6) U(r)
def compute_U_map(phi_high, c_map, Uth: float, mask) -> np.ndarray:
    """U(r) = U_Th · ln( Φ_high(r) / c(r) ).  c(r) is the LOW-bias calibration."""
    phi = np.asarray(phi_high, dtype=np.float64)
    if phi.shape != c_map.shape:
        raise ValueError("phi_high and c_map must have the same shape")
    U = np.full_like(phi, np.nan, dtype=np.float64)
    safe = mask & np.isfinite(c_map) & (c_map > 0) & (phi > 0)
    U[safe] = Uth * np.log(phi[safe] / c_map[safe])
    return U
 
 
# ----------------------------------------------------------------- 5) Vi (high)
def estimate_Vi_from_brightest(phi_high, U_map, mask, *,
                               bright_exclude_frac: float = 0.001,
                               min_valid_pixels: int = 50) -> float:
    """Cell terminal voltage at high bias = U(r) at the brightest region.
 
    Excludes the top `bright_exclude_frac` of pixels by EL intensity first
    (paper: brightest 0.1%) to suppress noise outliers, then takes max U.
    Set bright_exclude_frac=0 if you already cleaned outliers upstream.
    """
    safe = mask & np.isfinite(U_map) & np.isfinite(phi_high)
    if np.count_nonzero(safe) < min_valid_pixels:
        raise ValueError("too few valid pixels for Vi estimation")
    if bright_exclude_frac and bright_exclude_frac > 0:
        thr = np.quantile(phi_high[safe], 1.0 - bright_exclude_frac)
        keep = safe & (phi_high <= thr)
        if not keep.any():
            keep = safe
    else:
        keep = safe
    return float(np.nanmax(U_map[keep]))
 
 
# ----------------------------------------------------------------- 7) J(r)
def compute_J_map(J0_map, U_map, Uth: float, mask) -> np.ndarray:
    """J(r) = J0(r) · exp( U(r) / U_Th )."""
    if J0_map.shape != U_map.shape:
        raise ValueError("J0_map and U_map must have the same shape")
    J = np.full_like(U_map, np.nan, dtype=np.float64)
    safe = mask & np.isfinite(J0_map) & (J0_map > 0) & np.isfinite(U_map)
    J[safe] = J0_map[safe] * np.exp(U_map[safe] / Uth)
    return J
 
 
# ----------------------------------------------------------------- 8) Rs(r)
def compute_Rs_map(Vi: float, U_map, J_map, mask, *,
                   min_abs_J: float = 1e-30, drop_negative: bool = True) -> np.ndarray:
    """Rs(r) = (Vᵢ − U(r)) / J(r).   (Eq. 4)
 
    Returns the MAP. Do NOT divide by pixel count — the per-cell scalar is
    map_scalar(Rs_map, mask, 'mean'). Pixels with U>Vᵢ (negative Rs) are dropped.
    """
    Rs = np.full_like(U_map, np.nan, dtype=np.float64)
    safe = (mask & np.isfinite(U_map) & np.isfinite(J_map)
            & (np.abs(J_map) > min_abs_J))
    if drop_negative:
        safe &= (U_map <= float(Vi))
    Rs[safe] = (float(Vi) - U_map[safe]) / J_map[safe]
    return Rs
 
 
def compute_high_bias_maps(phi_high, c_map, J0_map, mask, Uth: float = 0.0259, *,
                           bright_exclude_frac: float = 0.001):
    """Steps 5-8 for one cell -> U, Vᵢ(high), J, Rs maps + scalar Rs,i."""
    U_map = compute_U_map(phi_high, c_map, Uth, mask)
    Vi_h = estimate_Vi_from_brightest(phi_high, U_map, mask,
                                      bright_exclude_frac=bright_exclude_frac)
    J_map = compute_J_map(J0_map, U_map, Uth, mask)
    Rs_map = compute_Rs_map(Vi_h, U_map, J_map, mask)
    return {"U_map": U_map, "Vi_high": Vi_h, "J_map": J_map,
            "Rs_map": Rs_map, "Rs_i": map_scalar(Rs_map, mask, "mean")}

## summary

In [5]:
# =============================================================== FULL MODULE
def load_module_cells(low_folder, high_folder, ext: str = ".tiff", *,
                      min_valid_intensity: float = 1e-6):
    """Load & mask every cell, pairing low/high by trailing index.
    Returns list of (phi_low, phi_high, mask) ordered by index."""
    import re
 
    def idx(p):
        m = re.findall(r"(\d+)", os.path.splitext(os.path.basename(p))[0])
        return int(m[-1]) if m else -1
 
    lows = sorted(glob.glob(os.path.join(low_folder, f"*{ext}")), key=idx)
    high_by_idx = {idx(hp): hp for hp in glob.glob(os.path.join(high_folder, f"*{ext}"))}
    cells = []
    for lp in lows:
        hp = high_by_idx.get(idx(lp))
        if hp is None:
            continue
        pl, ph, mask = mask_cell_pair(lp, hp, min_valid_intensity=min_valid_intensity)
        cells.append((pl, ph, mask))
    if not cells:
        raise ValueError("no matched low/high cell pairs found")
    return cells
 
 
def _dist(vals, prefix):
    """mean/median/min/max/std/n of a 1D array, keyed with a prefix."""
    v = np.asarray(vals, dtype=float)
    v = v[np.isfinite(v)]
    if v.size == 0:
        return {f"{prefix}_{k}": np.nan for k in ("mean", "median", "min", "max", "std")} \
               | {f"{prefix}_n": 0}
    return {f"{prefix}_mean": float(v.mean()), f"{prefix}_median": float(np.median(v)),
            f"{prefix}_min": float(v.min()), f"{prefix}_max": float(v.max()),
            f"{prefix}_std": float(v.std(ddof=1)) if v.size > 1 else 0.0,
            f"{prefix}_n": int(v.size)}
 
 
def _module_summary(rs_px, j0_px, u_px, Rs_i, J0_i, n_pix_i):
    """Build the module summary dict.
 
    Pixel-pooled distributions (every cell's pixels together) of Rs(r), J0(r),
    U(r); cell-level distributions of the 60 per-cell means; and the headline
    module predictions:
        Rs_module_total = Σ_i Rs_i   (cells in series -> resistances add)
        J0_module_avg   = mean_i J0_i
    vals_size = total valid pixels of the module (per-pixel -> module bridge).
    """
    Rs_i = np.asarray(Rs_i, float); J0_i = np.asarray(J0_i, float)
    out = {}
    out |= _dist(rs_px, "Rs_pixel")
    out |= _dist(j0_px, "J0_pixel")
    out |= _dist(u_px,  "U_pixel")
    out |= _dist(Rs_i,  "Rs_cell")        # spread across the 60 cells
    out |= _dist(J0_i,  "J0_cell")
    out["n_cells"] = int(np.sum(np.isfinite(Rs_i)))
    out["vals_size"] = int(np.sum(n_pix_i))           # total module pixels
    # headline module-level predictions
    out["Rs_module_total"] = float(np.nansum(Rs_i))   # Σ cell Rs (series)
    out["Rs_module_mean"]  = float(np.nanmean(Rs_i))
    out["J0_module_avg"]   = float(np.nanmean(J0_i))
    out["J0_module_median"] = float(np.nanmedian(J0_i))
    return out
 
 
def analyze_module(low_folder, high_folder, I_low, VT, *,
                   Uth: float = 0.0259, N: int = 60, ext: str = ".tiff",
                   cell_area_cm2=None, min_valid_intensity: float = 1e-6,
                   bright_exclude_frac: float = 0.001,
                   save_dir=None, module_id=None, n_rows: int = 6, n_cols: int = 10,
                   order: str = "row", save_J0_tiff=False, save_Rs_tiff=False):
    """Full pipeline for one module: Steps 1-8.
 
    Returns dict with f, Vi (low), Vi_high, per-cell c/J0/Rs maps, and the
    per-cell scalars J0_i / Rs_i. Optionally writes module-level J0 and Rs
    mosaics as 32-bit TIFFs (for ML) into save_dir as <module_id>_J0.tiff /
    <module_id>_Rs.tiff.
    """
    cells = load_module_cells(low_folder, high_folder, ext,
                              min_valid_intensity=min_valid_intensity)
    low_cells = [(pl, m) for pl, _ph, m in cells]
    f, S = compute_f(low_cells, I_low, VT, Uth, N=N, cell_area_cm2=cell_area_cm2)
    Vi = compute_Vi(S, f, I_low, Uth)
 
    c_maps, J0_maps, Rs_maps = [], [], []
    J0_i, Rs_i, Vi_high, n_pix_i = [], [], [], []
    pool_rs, pool_j0, pool_u = [], [], []      # pooled valid pixels for module stats
    for (pl, ph, m), vi in zip(cells, Vi):
        c_map, _ = compute_c_map(pl, vi, Uth, mask=m)
        J0_map = compute_J0_map(c_map, f, mask=m)
        hb = compute_high_bias_maps(ph, c_map, J0_map, m, Uth,
                                    bright_exclude_frac=bright_exclude_frac)
        c_maps.append(c_map)
        J0_maps.append(J0_map)
        Rs_maps.append(hb["Rs_map"])
        J0_i.append(map_scalar(J0_map, m, "mean"))
        Rs_i.append(hb["Rs_i"])
        Vi_high.append(hb["Vi_high"])
        n_pix_i.append(int(m.sum()))
        pool_rs.append(hb["Rs_map"][np.isfinite(hb["Rs_map"])])
        pool_j0.append(J0_map[np.isfinite(J0_map) & m])
        pool_u.append(hb["U_map"][np.isfinite(hb["U_map"]) & m])
 
    J0_i = np.asarray(J0_i)
    Rs_i = np.asarray(Rs_i)
    summary = _module_summary(np.concatenate(pool_rs) if pool_rs else np.array([]),
                              np.concatenate(pool_j0) if pool_j0 else np.array([]),
                              np.concatenate(pool_u) if pool_u else np.array([]),
                              Rs_i, J0_i, n_pix_i)
 
    result = {"f": f, "S": S, "Vi": Vi, "Vi_high": np.asarray(Vi_high),
              "c_maps": c_maps, "J0_maps": J0_maps, "Rs_maps": Rs_maps,
              "J0_i": J0_i, "Rs_i": Rs_i, "n_pix_i": n_pix_i,
              "summary": summary}
 
    if save_dir and module_id and (save_J0_tiff or save_Rs_tiff):
        os.makedirs(save_dir, exist_ok=True)
        if save_J0_tiff:
            mJ0 = assemble_module_map(J0_maps, n_rows, n_cols, order=order)
            save_module_map_tiff(mJ0, os.path.join(save_dir, f"{module_id}_J0.tiff"))
        if save_Rs_tiff:
            mRs = assemble_module_map(Rs_maps, n_rows, n_cols, order=order)
            save_module_map_tiff(mRs, os.path.join(save_dir, f"{module_id}_Rs.tiff"))
    return result
 
 
def save_feature_stack(J0_maps, Rs_maps, path_npy, *, n_rows: int = 6,
                       n_cols: int = 10, order: str = "row"):
    """Save a 2-channel module feature array (J0, Rs) as .npy for ML input.
    Shape = (2, H, W); channel 0 = J0 mosaic, channel 1 = Rs mosaic."""
    mJ0 = assemble_module_map(J0_maps, n_rows, n_cols, order=order)
    mRs = assemble_module_map(Rs_maps, n_rows, n_cols, order=order)
    np.save(path_npy, np.stack([mJ0, mRs], axis=0))
 
 
if __name__ == "__main__":
    print(__doc__)


el_blocks.py
 
Building blocks for module-level EL analysis (Rajput et al. 2018), to be called
from a larger run_module_analysis(). Three blocks so far:
 
    1. mask_cell_pair(low, high)  -> (phi_low, phi_high, mask)   [cv2 + Otsu]
    2. compute_f(low_cells, ...)  -> f, S        (one module constant)
    3. compute_Vi(S or cells, f)  -> Vi per cell
 
Formulae (your screenshots):
 
    f   = exp{ [ Σ_i (U_Th/2)·ln( I / S_i ) − V_T ] / ( (N/2)·U_Th ) }     (Eq.16)
    V_i = (U_Th/2)·ln( I / ( f · S_i ) )                                    (Eq.13)
    with  S_i = ∫ (1/Φ(r)) d²r  ≈  a_pixel · Σ_pixels (1/Φ_low)
 
The "30" in the paper's f is N/2 with N = 60, kept general here as N/2.
 
================================ IMPORTANT cv2 TRAP ============================
cv2.imread(path) WITHOUT a flag downconverts a 16-bit TIFF to 8-bit BGR. That
silently destroys the linear intensity scale the whole method depends on. Always
read with IMREAD_UNCHANGED (done in load_gray below).
 
Also: cv2'

## run module low cell

In [6]:

 
# ----------------------------------------------------------------- convenience
def load_module_low_cells(low_folder: str, ext: str = ".tiff", *,
                          high_folder: str | None = None,
                          min_valid_intensity: float = 1e-6):
    """Load & mask every low-bias cell crop in a folder.
 
    If high_folder is given, each cell's mask is the consistent low/high mask;
    otherwise the mask comes from the low image alone. Returns a list of
    (phi_low, mask) ordered by the trailing index in the filename.
    """
    def idx(p):
        import re
        m = re.findall(r"(\d+)", os.path.splitext(os.path.basename(p))[0])
        return int(m[-1]) if m else -1
 
    lows = sorted(glob.glob(os.path.join(low_folder, f"*{ext}")), key=idx)
    # low (..._20_NNN) and high (..._80_NNN) filenames differ -> pair by index
    high_by_idx = {}
    if high_folder:
        for hp in glob.glob(os.path.join(high_folder, f"*{ext}")):
            high_by_idx[idx(hp)] = hp
 
    cells = []
    for lp in lows:
        hp = high_by_idx.get(idx(lp)) if high_folder else None
        if hp:
            phi_low, _phi_high, mask = mask_cell_pair(
                lp, hp, min_valid_intensity=min_valid_intensity)
        else:
            phi_low = load_gray(lp)
            _, _, mask = mask_cell_pair(lp, lp,
                                        min_valid_intensity=min_valid_intensity)
        cells.append((phi_low, mask))
    return cells

## run module/run batch

In [7]:
#!/usr/bin/env python3
"""
run_modules.py
==============

Unity functions on top of el_blocks:

    single_module_analysis(...)  -> analyse ONE module: write Rs/J0 module TIFFs
                                    and return the summary row (stats + module
                                    totals + vals_size).
    batch_module_analysis(...)   -> loop every module in a CSV, skip the ones
                                    whose folders are missing, write one summary
                                    CSV at the end (+ per-module TIFFs).

Folder layout per module (cell crops already segmented):
    <root_dir>/<mid>_20/   60 low-bias  cell crops
    <root_dir>/<mid>_80/   60 high-bias cell crops
Module IDs (and optionally I_low/Isc and VT) come from columns of the CSV.
"""

from __future__ import annotations
import os
import csv
import logging
import numpy as np

import sys
eb = sys.modules['__main__']

#import el_blocks as eb

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
log = logging.getLogger("run_modules")


# ----------------------------------------------------------------- single
def single_module_analysis(
        module_id, root_dir, I_low, VT, *,
        Uth=0.0259, N=60,
        low_suffix="_20", high_suffix="_80", ext=".tiff",
        feature_dir=None, save_J0_tiff=True, save_Rs_tiff=True,
        n_rows=6, n_cols=10, order="row",
        bright_exclude_frac=0.001, cell_area_cm2=None,
        min_valid_intensity=1e-6, return_full=False):
    """Analyse one module. Returns a flat summary row (dict).

    If feature_dir is given, writes <feature_dir>/<module_id>_J0.tiff and
    _Rs.tiff (32-bit float module mosaics). Set return_full=True to also get
    the full result (maps, per-cell arrays).
    """
    low_folder = os.path.join(root_dir, f"{module_id}{low_suffix}")
    high_folder = os.path.join(root_dir, f"{module_id}{high_suffix}")

    res = eb.analyze_module(
        low_folder, high_folder, I_low, VT, Uth=Uth, N=N, ext=ext,
        cell_area_cm2=cell_area_cm2, min_valid_intensity=min_valid_intensity,
        bright_exclude_frac=bright_exclude_frac,
        save_dir=feature_dir, module_id=module_id, n_rows=n_rows, n_cols=n_cols,
        order=order,
        save_J0_tiff=bool(feature_dir) and save_J0_tiff,
        save_Rs_tiff=bool(feature_dir) and save_Rs_tiff)

    row = {"module": module_id, "I_low_A": I_low, "VT_V": VT,
           "f": res["f"], "status": "ok", "error": ""}
    row.update(res["summary"])
    return (row, res) if return_full else row


# ----------------------------------------------------------------- csv helpers
def _read_rows(path):
    with open(path, newline="") as fh:
        return list(csv.DictReader(fh))


def _num(row, col, default=None):
    if not col or col not in row or not str(row[col]).strip():
        return default
    try:
        return float(row[col])
    except (ValueError, TypeError):
        return default


def _write_csv(path, rows):
    if not rows:
        return
    keys = []
    for r in rows:
        for k in r:
            if k not in keys:
                keys.append(k)
    with open(path, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=keys)
        w.writeheader()
        w.writerows(rows)


# ----------------------------------------------------------------- batch
def batch_module_analysis(
        csv_path, root_dir, *,
        module_col="module",
        i_low_col=None, i_low_default=None,
        isc_col=None, isc_default=None, low_frac=0.20,
        vt_col=None, vt_default=0.0,
        Uth=0.0259, N=60,
        low_suffix="_20", high_suffix="_80", ext=".tiff",
        feature_dir=None, save_J0_tiff=True, save_Rs_tiff=True,
        n_rows=6, n_cols=10, order="row",
        bright_exclude_frac=0.001, cell_area_cm2=None,
        min_valid_intensity=1e-6,
        out_summary="module_summary.csv"):
    """Run every module listed in csv_path; skip missing; write summary CSV."""
    rows = _read_rows(csv_path)
    log.info("%d modules listed in %s", len(rows), os.path.basename(csv_path))
    if feature_dir:
        os.makedirs(feature_dir, exist_ok=True)

    summary_rows = []
    n_ok = n_skip = 0
    for r in rows:
        mid = str(r.get(module_col, "")).strip()
        if not mid:
            continue

        isc = _num(r, isc_col, isc_default)
        i_low = _num(r, i_low_col,
                     (low_frac * isc) if isc is not None else i_low_default)
        vt = _num(r, vt_col, vt_default)

        low_folder = os.path.join(root_dir, f"{mid}{low_suffix}")
        high_folder = os.path.join(root_dir, f"{mid}{high_suffix}")

        try:
            if not os.path.isdir(low_folder):
                raise FileNotFoundError(f"missing folder {low_folder}")
            if not os.path.isdir(high_folder):
                raise FileNotFoundError(f"missing folder {high_folder}")
            if i_low is None:
                raise ValueError("I_low unknown (set i_low_col/isc_col or defaults)")

            row = single_module_analysis(
                mid, root_dir, i_low, vt, Uth=Uth, N=N,
                low_suffix=low_suffix, high_suffix=high_suffix, ext=ext,
                feature_dir=feature_dir, save_J0_tiff=save_J0_tiff,
                save_Rs_tiff=save_Rs_tiff, n_rows=n_rows, n_cols=n_cols,
                order=order, bright_exclude_frac=bright_exclude_frac,
                cell_area_cm2=cell_area_cm2, min_valid_intensity=min_valid_intensity)
            n_ok += 1
            log.info("%-26s  Rs_total=%.3e  J0_avg=%.3e  (%d cells, %d px)",
                     mid, row["Rs_module_total"], row["J0_module_avg"],
                     row["n_cells"], row["vals_size"])
        except Exception as exc:                       # noqa: BLE001
            n_skip += 1
            row = {"module": mid, "I_low_A": i_low, "VT_V": vt,
                   "status": "error", "error": str(exc)}
            log.warning("%-26s  SKIPPED: %s", mid, exc)
        summary_rows.append(row)

    _write_csv(out_summary, summary_rows)
    log.info("done: %d ok, %d skipped -> %s", n_ok, n_skip, out_summary)
    return summary_rows




# Test run

In [8]:



row, res = single_module_analysis(
    "3903_35_1_01272020", r"C:\Users\Ghozy Abror\OneDrive - Institut Teknologi Bandung\Karirku\UNSW\Thesis\Coding\EL_cell_low_res_2",
    I_low=1.0, VT=30.5,
    return_full=True,save_J0_tiff=False, save_Rs_tiff=False
)

print(res.keys())        # inspect available maps / per-cell arrays
print("f =", res["f"])
print(res["summary"])    # the per-module summary that got flattened into `row`

dict_keys(['f', 'S', 'Vi', 'Vi_high', 'c_maps', 'J0_maps', 'Rs_maps', 'J0_i', 'Rs_i', 'n_pix_i', 'summary'])
f = 1.9510657433521363e-20
{'Rs_pixel_mean': 930.1588405490959, 'Rs_pixel_median': 918.5461060721229, 'Rs_pixel_min': 0.0, 'Rs_pixel_max': 331694.59188815445, 'Rs_pixel_std': 431.47457453439006, 'Rs_pixel_n': 3382543, 'J0_pixel_mean': 5.324864864978545e-14, 'J0_pixel_median': 5.1470933029725456e-14, 'J0_pixel_min': 3.72506520867637e-14, 'J0_pixel_max': 1.3157992363535413e-13, 'J0_pixel_std': 6.964173663532925e-15, 'J0_pixel_n': 3382543, 'U_pixel_mean': 0.49908746327477205, 'U_pixel_median': 0.4990002992180517, 'U_pixel_min': 0.3952537601447679, 'U_pixel_max': 0.5156763215520574, 'U_pixel_std': 0.003491386787904509, 'U_pixel_n': 3382543, 'Rs_cell_mean': 930.024218623446, 'Rs_cell_median': 930.0624586970539, 'Rs_cell_min': 710.5783955660069, 'Rs_cell_max': 1193.3949789522815, 'Rs_cell_std': 89.19735844815538, 'Rs_cell_n': 60, 'J0_cell_mean': 5.3247990325371804e-14, 'J0_cell_median

## RUNNNNN

In [9]:
# ----------------------------------------------------------------- example
if __name__ == "__main__":
    batch_module_analysis(
         csv_path = "AnonDB_Zub_72_hanya36.csv",
         root_dir = r"C:\Users\Ghozy Abror\OneDrive - Institut Teknologi Bandung\Karirku\UNSW\Thesis\Coding\EL_cell_low_res_2",
         module_col="module_code",
         i_low_col="Low_Applied_Current_(A)",     # or isc_col="Isc" (-> I_low = 0.2*Isc)
         vt_col="Low_Applied_Voltage_(V)",
         N=72,                     # <-- jumlah sel per modul, dipakai di rumus f (Eq. 16)
         n_rows=6, n_cols=12,      # <-- susunan mosaik untuk TIFF Rs/J0 (6x12 = 72)
         feature_dir=r"C:\Users\Ghozy Abror\OneDrive - Institut Teknologi Bandung\Karirku\UNSW\Thesis\Coding\J0_Rs72_hanya36",   # writes <mid>_J0.tiff / _Rs.tiff
         out_summary=r"C:\Users\Ghozy Abror\OneDrive - Institut Teknologi Bandung\Karirku\UNSW\Thesis\Coding\module_summary72_hanya36.csv",
     )
    raise SystemExit("Call batch_module_analysis(...) — see commented example.")

INFO | 2 modules listed in AnonDB_Zub_72_hanya36.csv
INFO | 2532_36_1_08062019          Rs_total=5.434e+04  J0_avg=2.171e-16  (72 cells, 12038052 px)
INFO | 2533_36_1_08062019          Rs_total=5.442e+04  J0_avg=1.429e-16  (72 cells, 12043180 px)
INFO | done: 2 ok, 0 skipped -> C:\Users\Ghozy Abror\OneDrive - Institut Teknologi Bandung\Karirku\UNSW\Thesis\Coding\module_summary72_hanya36.csv


SystemExit: Call batch_module_analysis(...) — see commented example.

c:\Users\Ghozy Abror\anaconda3\envs\UNSWThesis\lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [15]:
# !/usr/bin/env python3
"""
validate_batch.py
=================

Validasi per-step (Step 1-8) untuk seluruh modul di CSV — READ-ONLY.

Tidak menulis/menimpa apa pun yang sudah ada:
  * tidak menyentuh <mid>_J0.tiff / <mid>_Rs.tiff
  * tidak menimpa module_summary.csv
Hanya MEMBACA citra sel, menghitung ulang di memori, lalu menulis file BARU:

  1. <out_dir>/percell_values.csv    satu baris = satu sel
        module, cell, Vi_low, Vi_high, c_i, U_mean, J_mean, J0_i, Rs_i, n_pix
     -> ini yang dipakai untuk semua validasi & Step 9 (PVLIB/pySpice) nanti.

  2. <out_dir>/validation_summary.csv   satu baris = satu modul
        cek per-step: sum(Vi_low) vs VT, sum(Vi_high) vs VT_high, dll.

  3. <out_dir>/plots/*.png          grafik validasi tiap step

Validasi yang dijalankan (sesuai tabelmu)
-----------------------------------------
Step 2  sum(Vi_low)  == VT_low        -> residual harus ~0 (identitas Eq.16/13)
Step 3  c_i vs cell number, c_i vs Vi -> sebaran ci (paper Fig.4)
Step 5  sum(Vi_high) == VT_high       -> hanya jika VT_high diberikan
Step 6  sum(mean U per cell) vs VT_high
Step 7  J efektif modul vs I_high     -> J(r) rata-rata * area vs arus terukur
Step 8  Rs distribusi (paper Fig.7)
Step 4  J0 distribusi (paper Fig.7)

CATATAN Step 2: identitas sum(Vi)==VT bersifat MATEMATIS (f diturunkan justru
supaya ini berlaku). Jadi residual ~0 memvalidasi implementasi, bukan fisika.
Yang benar-benar menguji fisika adalah Step 5/6 (bias tinggi), karena VT_high
TIDAK dipakai saat menghitung apa pun -> ini prediksi independen.
"""

from __future__ import annotations
import os
import csv
import logging
import numpy as np

import matplotlib
matplotlib.use("Agg")            # aman untuk batch / headless
import matplotlib.pyplot as plt

#import el_blocks as eb

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
log = logging.getLogger("validate")


# --------------------------------------------------------------- helpers
def _num(row, col, default=None):
    if not col or col not in row or not str(row[col]).strip():
        return default
    try:
        return float(row[col])
    except (ValueError, TypeError):
        return default


def _write_csv(path, rows):
    if not rows:
        return
    keys = []
    for r in rows:
        for k in r:
            if k not in keys:
                keys.append(k)
    with open(path, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=keys)
        w.writeheader()
        w.writerows(rows)


# --------------------------------------------------------------- inti: 1 modul
def validate_single_module(module_id, root_dir, I_low, VT_low, *,
                           VT_high=None, I_high=None,
                           Uth=0.0259, N=60,
                           low_suffix="_20", high_suffix="_80", ext=".tiff",
                           bright_exclude_frac=0.001,
                           min_valid_intensity=1e-6):
    """Hitung ulang Step 1-8 untuk satu modul (di memori) dan kembalikan
    (per_cell_rows, validation_row). Tidak menulis file apa pun."""
    low_folder = os.path.join(root_dir, f"{module_id}{low_suffix}")
    high_folder = os.path.join(root_dir, f"{module_id}{high_suffix}")

    cells = eb.load_module_cells(low_folder, high_folder, ext,
                                 min_valid_intensity=min_valid_intensity)
    low_cells = [(pl, m) for pl, _ph, m in cells]

    # --- Step 1: f
    f, S = eb.compute_f(low_cells, I_low, VT_low, Uth, N=N)
    # --- Step 2: Vi low
    Vi = eb.compute_Vi(S, f, I_low, Uth)

    per_cell = []
    Vi_high_l, c_i_l, U_mean_l, J_mean_l, J0_i_l, Rs_i_l, npix_l = ([] for _ in range(7))

    for k, ((pl, ph, m), vi) in enumerate(zip(cells, Vi)):
        # --- Step 3: c(r)
        c_map, _ = eb.compute_c_map(pl, vi, Uth, mask=m)
        c_i = eb.map_scalar(c_map, m, "mean")
        # --- Step 4: J0(r)
        J0_map = eb.compute_J0_map(c_map, f, mask=m)
        J0_i = eb.map_scalar(J0_map, m, "mean")
        # --- Step 6: U(r)  (butuh c(r) + citra bias tinggi)
        U_map = eb.compute_U_map(ph, c_map, Uth, m)
        U_mean = eb.map_scalar(U_map, m, "mean")
        # --- Step 5: Vi high
        vi_h = eb.estimate_Vi_from_brightest(ph, U_map, m,
                                             bright_exclude_frac=bright_exclude_frac)
        # --- Step 7: J(r)
        J_map = eb.compute_J_map(J0_map, U_map, Uth, m)
        J_mean = eb.map_scalar(J_map, m, "mean")
        # --- Step 8: Rs(r)
        Rs_map = eb.compute_Rs_map(vi_h, U_map, J_map, m)
        Rs_i = eb.map_scalar(Rs_map, m, "mean")

        n_pix = int(m.sum())
        per_cell.append({"module": module_id, "cell": k,
                         "Vi_low": float(vi), "Vi_high": float(vi_h),
                         "c_i": c_i, "U_mean": U_mean, "J_mean": J_mean,
                         "J0_i": J0_i, "Rs_i": Rs_i, "n_pix": n_pix})
        Vi_high_l.append(vi_h); c_i_l.append(c_i); U_mean_l.append(U_mean)
        J_mean_l.append(J_mean); J0_i_l.append(J0_i); Rs_i_l.append(Rs_i)
        npix_l.append(n_pix)

    Vi_high = np.asarray(Vi_high_l); U_mean = np.asarray(U_mean_l)
    J_mean = np.asarray(J_mean_l); Rs_i = np.asarray(Rs_i_l); J0_i = np.asarray(J0_i_l)

    # ---------------- validasi per step ----------------
    v = {"module": module_id, "n_cells": len(cells), "f": f,
         "I_low": I_low, "VT_low": VT_low, "VT_high": VT_high, "I_high": I_high,
         "status": "ok", "error": ""}

    # Step 2: identitas matematis sum(Vi_low) == VT_low
    sum_vi = float(np.sum(Vi))
    v["step2_sum_Vi_low"] = sum_vi
    v["step2_residual_V"] = sum_vi - VT_low
    v["step2_pass"] = bool(abs(sum_vi - VT_low) < 1e-6)

    # Step 3: sebaran c_i (paper: variasi ci antar sel itu wajar & penting)
    c_arr = np.asarray(c_i_l, float)
    v["step3_ci_mean"] = float(np.nanmean(c_arr))
    v["step3_ci_std"] = float(np.nanstd(c_arr, ddof=1)) if c_arr.size > 1 else 0.0
    v["step3_ci_cv"] = (v["step3_ci_std"] / v["step3_ci_mean"]
                        if v["step3_ci_mean"] else np.nan)

    # Step 5: sum(Vi_high) vs VT_high  -- PREDIKSI INDEPENDEN (VT_high tak dipakai)
    sum_vih = float(np.sum(Vi_high))
    v["step5_sum_Vi_high"] = sum_vih
    if VT_high is not None:
        v["step5_residual_V"] = sum_vih - VT_high
        v["step5_rel_error"] = (sum_vih - VT_high) / VT_high
        v["step5_pass"] = bool(abs(v["step5_rel_error"]) < 0.05)   # toleransi 5%
    else:
        v["step5_residual_V"] = np.nan
        v["step5_rel_error"] = np.nan
        v["step5_pass"] = None

    # Step 6: sum(mean U per cell) vs VT_high
    sum_u = float(np.sum(U_mean))
    v["step6_sum_U_mean"] = sum_u
    if VT_high is not None:
        v["step6_residual_V"] = sum_u - VT_high
        v["step6_rel_error"] = (sum_u - VT_high) / VT_high
    else:
        v["step6_residual_V"] = np.nan
        v["step6_rel_error"] = np.nan

    # Step 7: J efektif modul vs I_high
    # J(r) per-piksel; arus sel ~ sum(J)*a_pixel. Karena a_pixel=1 (per-piksel),
    # J_cell_effective = mean(J)*n_pix. Sel seri -> arus modul = rata-rata antar sel.
    J_cell_current = J_mean * np.asarray(npix_l, float)
    v["step7_J_cell_current_mean"] = float(np.nanmean(J_cell_current))
    v["step7_J_cell_current_std"] = float(np.nanstd(J_cell_current, ddof=1)) \
        if J_cell_current.size > 1 else 0.0
    if I_high:
        v["step7_ratio_to_I_high"] = v["step7_J_cell_current_mean"] / I_high
    else:
        v["step7_ratio_to_I_high"] = np.nan

    # Step 4 & 8: distribusi J0 dan Rs (paper Fig.7)
    v["step8_Rs_mean"] = float(np.nanmean(Rs_i))
    v["step8_Rs_std"] = float(np.nanstd(Rs_i, ddof=1)) if Rs_i.size > 1 else 0.0
    v["step8_Rs_total"] = float(np.nansum(Rs_i))
    v["step8_Rs_median"] = float(np.nanmedian(Rs_i))
    v["step8_Rs_negatives"] = int(np.sum(Rs_i < 0))
    v["step4_J0_mean"] = float(np.nanmean(J0_i))
    v["step4_J0_std"] = float(np.nanstd(J0_i, ddof=1)) if J0_i.size > 1 else 0.0
    v["step4_J0_median"] = float(np.nanmedian(J0_i))

    return per_cell, v


# --------------------------------------------------------------- grafik
def _plots(per_cell_rows, val_rows, plot_dir):
    os.makedirs(plot_dir, exist_ok=True)
    if not per_cell_rows:
        return
    mods = sorted({r["module"] for r in per_cell_rows})
    arr = lambda rows, k: np.array([r[k] for r in rows], float)  # noqa: E731

    # --- Step 2: residual sum(Vi) - VT (harus ~0) ---
    ok = [v for v in val_rows if v.get("status") == "ok"]
    if ok:
        res = arr(ok, "step2_residual_V")
        plt.figure(figsize=(7, 4))
        plt.hist(res, bins=40)
        plt.xlabel("sum(Vi_low) - VT_low  [V]")
        plt.ylabel("jumlah modul")
        plt.title("Step 2: identitas sum(Vi) = VT (harus ~0)")
        plt.tight_layout(); plt.savefig(os.path.join(plot_dir, "step2_sumVi_residual.png"), dpi=120)
        plt.close()

    # --- Step 3: c_i vs nomor sel, dan c_i vs Vi (paper Fig.4) ---
    sample = mods[:3]
    plt.figure(figsize=(8, 4.5))
    for m in sample:
        rows = [r for r in per_cell_rows if r["module"] == m]
        plt.plot(arr(rows, "cell"), arr(rows, "c_i"), "o", ms=4, label=m, alpha=.7)
    plt.xlabel("nomor sel"); plt.ylabel("c_i (calibration constant)")
    plt.title("Step 3: distribusi c_i per sel (bandingkan paper Fig.4)")
    plt.legend(fontsize=7); plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, "step3_ci_vs_cell.png"), dpi=120); plt.close()

    plt.figure(figsize=(6, 4.5))
    plt.scatter(arr(per_cell_rows, "Vi_low"), arr(per_cell_rows, "c_i"), s=6, alpha=.3)
    plt.xlabel("Vi_low [V]"); plt.ylabel("c_i")
    plt.title("Step 3: c_i vs Vi_low (semua sel)")
    plt.tight_layout(); plt.savefig(os.path.join(plot_dir, "step3_ci_vs_Vi.png"), dpi=120)
    plt.close()

    # --- Step 5: sum(Vi_high) vs VT_high (prediksi independen) ---
    ok5 = [v for v in ok if v.get("VT_high") not in (None, "")
           and np.isfinite(v.get("step5_residual_V", np.nan))]
    if ok5:
        x = arr(ok5, "VT_high"); y = arr(ok5, "step5_sum_Vi_high")
        plt.figure(figsize=(5.5, 5.5))
        plt.scatter(x, y, s=18, alpha=.6)
        lo, hi = min(x.min(), y.min()), max(x.max(), y.max())
        plt.plot([lo, hi], [lo, hi], "k--", lw=1, label="y = x")
        plt.xlabel("VT_high terukur [V]"); plt.ylabel("sum(Vi_high) prediksi [V]")
        plt.title("Step 5: prediksi VT bias tinggi vs terukur")
        plt.legend(); plt.tight_layout()
        plt.savefig(os.path.join(plot_dir, "step5_VT_high_parity.png"), dpi=120); plt.close()

    # --- Step 6: sum(U_mean) vs VT_high ---
    ok6 = [v for v in ok if np.isfinite(v.get("step6_residual_V", np.nan))]
    if ok6:
        x = arr(ok6, "VT_high"); y = arr(ok6, "step6_sum_U_mean")
        plt.figure(figsize=(5.5, 5.5))
        plt.scatter(x, y, s=18, alpha=.6, color="tab:orange")
        lo, hi = min(x.min(), y.min()), max(x.max(), y.max())
        plt.plot([lo, hi], [lo, hi], "k--", lw=1, label="y = x")
        plt.xlabel("VT_high terukur [V]"); plt.ylabel("sum(mean U per sel) [V]")
        plt.title("Step 6: U(r) rata-rata vs VT bias tinggi")
        plt.legend(); plt.tight_layout()
        plt.savefig(os.path.join(plot_dir, "step6_U_vs_VT_high.png"), dpi=120); plt.close()

    # --- Step 4 & 8: distribusi J0 dan Rs per sel (paper Fig.7) ---
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    rs = arr(per_cell_rows, "Rs_i"); j0 = arr(per_cell_rows, "J0_i")
    rs = rs[np.isfinite(rs)]; j0 = j0[np.isfinite(j0)]
    if rs.size:
        ax[0].hist(rs[rs < np.percentile(rs, 99)], bins=60)
    ax[0].set_xlabel("Rs per sel"); ax[0].set_ylabel("jumlah sel")
    ax[0].set_title("Step 8: distribusi Rs (99th pct clip)")
    if j0.size:
        ax[1].hist(j0[j0 < np.percentile(j0, 99)], bins=60, color="tab:red")
    ax[1].set_xlabel("J0 per sel"); ax[1].set_title("Step 4: distribusi J0 (99th pct clip)")
    plt.tight_layout(); plt.savefig(os.path.join(plot_dir, "step4_8_Rs_J0_hist.png"), dpi=120)
    plt.close()

    # --- Step 7: arus sel efektif vs I_high ---
    ok7 = [v for v in ok if np.isfinite(v.get("step7_ratio_to_I_high", np.nan))]
    if ok7:
        plt.figure(figsize=(7, 4))
        plt.hist(arr(ok7, "step7_ratio_to_I_high"), bins=40, color="tab:green")
        plt.axvline(1.0, color="k", ls="--", lw=1, label="rasio = 1")
        plt.xlabel("J efektif sel / I_high"); plt.ylabel("jumlah modul")
        plt.title("Step 7: arus rekonstruksi vs arus injeksi")
        plt.legend(); plt.tight_layout()
        plt.savefig(os.path.join(plot_dir, "step7_J_vs_Ihigh.png"), dpi=120); plt.close()

    log.info("grafik disimpan di %s", plot_dir)


# --------------------------------------------------------------- batch
def validate_batch(csv_path, root_dir, out_dir, *,
                   module_col="module",
                   i_low_col=None, i_low_default=None,
                   isc_col=None, isc_default=None, low_frac=0.20,
                   vt_low_col=None, vt_low_default=None,
                   vt_high_col=None, vt_high_default=None,
                   i_high_col=None, i_high_default=None, high_frac=0.80,
                   Uth=0.0259, N=60,
                   low_suffix="_20", high_suffix="_80", ext=".tiff",
                   bright_exclude_frac=0.001, min_valid_intensity=1e-6,
                   make_plots=True, limit=None):
    """Jalankan validasi read-only untuk semua modul di csv_path.

    Menulis HANYA ke out_dir (folder baru):
        percell_values.csv, validation_summary.csv, plots/*.png
    """
    os.makedirs(out_dir, exist_ok=True)
    with open(csv_path, newline="") as fh:
        rows = list(csv.DictReader(fh))
    if limit:
        rows = rows[:limit]
    log.info("%d modul akan divalidasi (read-only)", len(rows))

    per_cell_all, val_all = [], []
    n_ok = n_skip = 0
    for r in rows:
        mid = str(r.get(module_col, "")).strip()
        if not mid:
            continue
        isc = _num(r, isc_col, isc_default)
        i_low = _num(r, i_low_col,
                     (low_frac * isc) if isc is not None else i_low_default)
        i_high = _num(r, i_high_col,
                      (high_frac * isc) if isc is not None else i_high_default)
        vt_low = _num(r, vt_low_col, vt_low_default)
        vt_high = _num(r, vt_high_col, vt_high_default)

        try:
            if i_low is None:
                raise ValueError("I_low tidak diketahui")
            if vt_low is None:
                raise ValueError("VT_low tidak diketahui (wajib untuk f)")
            pc, v = validate_single_module(
                mid, root_dir, i_low, vt_low, VT_high=vt_high, I_high=i_high,
                Uth=Uth, N=N, low_suffix=low_suffix, high_suffix=high_suffix,
                ext=ext, bright_exclude_frac=bright_exclude_frac,
                min_valid_intensity=min_valid_intensity)
            # bawa metadata tambahan (mis. make_model) kalau ada di CSV
            for extra in ("make_model", "Rs", "I0", "Isc", "Voc", "Pmp"):
                if extra in r:
                    v[extra] = r[extra]
            per_cell_all.extend(pc)
            val_all.append(v)
            n_ok += 1
            log.info("%-26s  step2_res=%.2e  Rs_tot=%.3e  J0_avg=%.3e",
                     mid, v["step2_residual_V"], v["step8_Rs_total"], v["step4_J0_mean"])
        except Exception as exc:                     # noqa: BLE001
            n_skip += 1
            val_all.append({"module": mid, "status": "error", "error": str(exc)})
            log.warning("%-26s  SKIPPED: %s", mid, exc)

    pc_path = os.path.join(out_dir, "percell_values.csv")
    val_path = os.path.join(out_dir, "validation_summary.csv")
    _write_csv(pc_path, per_cell_all)
    _write_csv(val_path, val_all)
    log.info("tulis %s (%d baris sel)", pc_path, len(per_cell_all))
    log.info("tulis %s (%d modul)", val_path, len(val_all))

    if make_plots:
        _plots(per_cell_all, val_all, os.path.join(out_dir, "plots"))

    # ringkasan lulus/gagal
    ok = [v for v in val_all if v.get("status") == "ok"]
    if ok:
        p2 = sum(1 for v in ok if v.get("step2_pass"))
        log.info("Step 2 (identitas sum Vi = VT): %d/%d lulus", p2, len(ok))
        p5 = [v for v in ok if v.get("step5_pass") is not None]
        if p5:
            log.info("Step 5 (VT bias tinggi, toleransi 5%%): %d/%d lulus",
                     sum(1 for v in p5 if v["step5_pass"]), len(p5))
    log.info("selesai: %d ok, %d dilewati", n_ok, n_skip)
    return per_cell_all, val_all


#if __name__ == "__main__":
 #   raise SystemExit("Panggil validate_batch(...) dari notebook.")

In [16]:
#from validate_batch import validate_batch

pc, val = validate_batch(
    csv_path = "AnonDB_zub_60.csv",
    root_dir = r"C:\Users\Ghozy Abror\OneDrive - Institut Teknologi Bandung\Karirku\UNSW\Thesis\Coding\EL_cell_low_res_2",
    out_dir  = r"C:\Users\Ghozy Abror\OneDrive - Institut Teknologi Bandung\Karirku\UNSW\Thesis\Coding\validation",     # folder BARU, tidak menimpa apa pun
    module_col  = "module_code",              # <-- ini kuncinya
    i_low_col   = "Low_Applied_Current_(A)",  # I_low asli, bukan 0.2*Isc
    vt_low_col  = "Low_Applied_Voltage_(V)",
    i_high_col  = "High_Applied_Current_(A)",
    vt_high_col = "High_Applied_Voltage_(V)"# -> Step 5 & 6 jadi hidup
    #limit = 5                  # coba 5 modul dulu sebelum full run
)

INFO | 437 modul akan divalidasi (read-only)
WARNING | 3395_30_1_03182016          SKIPPED: no matched low/high cell pairs found
WARNING | 3394_30_1_03182016          SKIPPED: no matched low/high cell pairs found
WARNING | 3411_30_1_03182016          SKIPPED: no matched low/high cell pairs found
WARNING | 3199_30_4_12162015          SKIPPED: no matched low/high cell pairs found
WARNING | 3197_30_4_12162015          SKIPPED: no matched low/high cell pairs found
WARNING | 3159_30_4_12222015          SKIPPED: no matched low/high cell pairs found
WARNING | 3160_30_4_12222015          SKIPPED: no matched low/high cell pairs found
WARNING | 3195_30_4_12162015          SKIPPED: no matched low/high cell pairs found
WARNING | 3396_30_1_04042016          SKIPPED: no matched low/high cell pairs found
WARNING | 3201_30_4_12162015          SKIPPED: no matched low/high cell pairs found
WARNING | 3255_5_6_07262017           SKIPPED: no matched low/high cell pairs found
WARNING | 3255_5_6_01172020    

In [ ]:
batch_module_analysis(
         csv_path = "AnonDB_zub_60.csv",
         root_dir = r"C:\Users\Ghozy Abror\OneDrive - Institut Teknologi Bandung\Karirku\UNSW\Thesis\Coding\EL_cell_low_res_2",
         module_col="module_code",
         i_low_col="Low_Applied_Current_(A)",     # or isc_col="Isc" (-> I_low = 0.2*Isc)
         vt_col="Low_Applied_Voltage_(V)",
         feature_dir=r"C:\Users\Ghozy Abror\OneDrive - Institut Teknologi Bandung\Karirku\UNSW\Thesis\Coding\J0_Rs",   # writes <mid>_J0.tiff / _Rs.tiff
         out_summary=r"C:\Users\Ghozy Abror\OneDrive - Institut Teknologi Bandung\Karirku\UNSW\Thesis\Coding\module_summary.csv",
     )

In [14]:
import csv
with open("AnonDB_zub_60.csv", newline="") as fh:
    reader = csv.DictReader(fh)
    print(reader.fieldnames)          # <-- nama kolom yang sebenarnya
    print(next(reader))               # <-- contoh satu baris

['Mod_ID', 'Confidential', 'Make', 'Model', 'Interconnect_Tech', 'Module_Area_(cm2)', 'Junction_Box_Type', 'Cell_Wafer_Type', 'Cell_Tech', 'Cell_Area_(cm2)', 'Num_Cells', 'Total_Exposure', 'Nameplate_Isc_(A)', 'Nameplate_Voc_(V)', 'Nameplate_Imp_(A)', 'Nameplate_Vmp_(V)', 'Nameplate_Pmp_(W)', 'Isc_(A)', 'Voc_(V)', 'Imp_(A)', 'Vmp_(V)', 'Pmp_(W)', 'FF_(percent)', 'Measured_Temperature_(C)', 'Temp_Measurement_Method', 'Voltage_Temperature_Coefficient_(mV/C)', 'Simulator_Make', 'Simulator_Model', 'IV_Date', 'IV_Time', 'IV_Lab_Location', 'Camera_Make', 'Camera_Model', 'Detector_Type', 'Image_Resolution_(MP)', 'Longpass_Filter_Wavelength_(nm)', 'High_Applied_Current_(A)', 'High_Applied_Voltage_(V)', 'High_Sensor_Exposure_Time_(s)', 'Low_Applied_Current_(A)', 'Applied_Current_Diff', 'Low_Applied_Voltage_(V)', 'Applied_Voltage_Diff', 'Low_Sensor_Exposure_Time_(s)', 'Sensor_Exp_Diff', 'ISO', 'Aperture', 'High_Temperature_(C)', 'Low_Temperature_(C)', 'Working_Distance_(m)', 'High_EL_Date', 'Hig